# core

> Fill in a module description here

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp core

## Database

In [ ]:
#| export
class Video: id:int; title:str=''; overview:str=''; transcript:str=''; length:int=0; sample_rate:int=0
class Frame: id:int; video_id:int; frame_number:int; subtitle:str=''
class Run: id:int; deploy_time:str; finish_time:str; request_timer:str; video_id:int; model:str; usage:str; num_frames:int; description:str
class RunFrame: run_id:int; frame_id:int; type:str; system_prompt:str; prompt:str; description:str; usage:str

In [ ]:
#| export
from fastlite import *

In [ ]:
!rm db.db
db = database('db.db'); db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
??Queryable.schema

Type:        property
String form: <property object>
Source:     
# Queryable.schema.fget
@property
def schema(self) -> str:
    "SQL schema for this table or view."
    return self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0]

In [ ]:
@patch(as_prop=True)
def schema(self:Queryable) -> str:
    "SQL schema for this table or view."
    return hl_md(self.db.execute(
        "select sql from sqlite_master where name = ?", (self.name,)
    ).fetchone()[0], lang='sql')

In [ ]:
videos = db.create(Video, transform=True); videos.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER
)
```

</div>

In [ ]:
frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')]); frames.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [frame] (
   [id] INTEGER PRIMARY KEY,
   [video_id] INTEGER REFERENCES [video]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_number] INTEGER,
   [subtitle] TEXT
)
```

</div>

In [ ]:
runs = db.create(Run, transform=True); runs.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [run] (
   [id] INTEGER PRIMARY KEY,
   [deploy_time] TEXT,
   [finish_time] TEXT,
   [request_timer] TEXT,
   [video_id] INTEGER,
   [model] TEXT,
   [usage] TEXT,
   [num_frames] INTEGER,
   [description] TEXT
)
```

</div>

In [ ]:
runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True); runframes.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [run_frame] (
   [run_id] INTEGER REFERENCES [run]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [frame_id] INTEGER REFERENCES [frame]([id]) ON UPDATE CASCADE ON DELETE CASCADE,
   [type] TEXT,
   [system_prompt] TEXT,
   [prompt] TEXT,
   [description] TEXT,
   [usage] TEXT,
   PRIMARY KEY ([run_id], [frame_id], [type])
)
```

</div>

In [ ]:
#| export
from typing import NamedTuple
from apswutils.db import Database, Table
class DBResources(NamedTuple): db:Database; videos:Table; frames:Table; runs:Table; runframes:Table

In [ ]:
#| export
from fastcore.all import *
def init_db(
    path:str|Path='db.db' # Path to database
) -> DBResources:
    "Initialize a database and return the database, as well as its videos, frames, runs, and runframes tables."
    db = database(path)
    videos = db.create(Video, transform=True)
    frames = db.create(Frame, transform=True, foreign_keys=[('video_id', 'video', 'id')])
    runs = db.create(Run, transform=True)
    runframes = db.create(RunFrame, pk=['run_id', 'frame_id', 'type'], foreign_keys=[('run_id', 'run', 'id'), ('frame_id', 'frame', 'id')], transform=True)
    return DBResources(db, videos, frames, runs, runframes)


In [ ]:
!rm db.db
o = init_db()

In [ ]:
o.db

<Database <apsw.Connection "/app/data/vlm-monitor/nbs/db.db">>

In [ ]:
o.videos.schema

<div class="prose" markdown="1">

```sql
CREATE TABLE [video] (
   [id] INTEGER PRIMARY KEY,
   [title] TEXT,
   [overview] TEXT,
   [transcript] TEXT,
   [length] INTEGER,
   [sample_rate] INTEGER
)
```

</div>

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()